# Attention Visualization in Transformers

Attention weights show how much each token attends to every other token during processing. While attention weights are not strict feature attributions, they provide insight into which parts of the input the model considers when producing each output.

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Load pretrained model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)

# Set model to evaluation mode
model.eval()

In [ ]:
# Tokenize sample sentence
sentence = "The bank by the river was flooded after heavy rain."
inputs = tokenizer.encode(sentence, return_tensors="pt")

# Get model outputs with attention
with torch.no_grad():
    outputs = model(inputs)

# Extract attention weights
attention = outputs[-1]  # List of attention tensors from each layer
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

print(f"Sentence: {sentence}")
print(f"Tokens: {tokens}")
print(f"Number of layers: {len(attention)}")
print(f"Attention shape (layer, batch, heads, seq_len, seq_len): {attention[0].shape}")

In [ ]:
# Extract attention from last layer, first head
last_layer_attention = attention[-1][0]  # Remove batch dimension
first_head_attention = last_layer_attention[0].cpu().numpy()  # First head

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(first_head_attention, 
            xticklabels=tokens, 
            yticklabels=tokens, 
            cmap='viridis',
            cbar_kws={'label': 'Attention Weight'})
plt.title("Attention Weights - Last Layer, Head 1", fontsize=14)
plt.xlabel("Attending to (Key)", fontsize=12)
plt.ylabel("Attending from (Query)", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Plot multiple heads from last layer
num_heads = 6
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for head_idx in range(num_heads):
    head_attention = last_layer_attention[head_idx].cpu().numpy()
    
    sns.heatmap(head_attention,
                ax=axes[head_idx],
                xticklabels=tokens,
                yticklabels=tokens,
                cmap='viridis',
                cbar=False)
    axes[head_idx].set_title(f"Head {head_idx + 1}", fontsize=12)
    axes[head_idx].set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    axes[head_idx].set_yticklabels(tokens, fontsize=9)

fig.suptitle("Attention Patterns - Last Layer (6 Heads)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## Limitations

Attention weights show correlations but not necessarily causal importance. Different heads may focus on syntactic, positional, or semantic patterns. Attention is not a faithful explanation of the model's reasoning — methods like Integrated Gradients are more principled for attributions.